In [4]:
import mysql.connector

try:
    # Kết nối tới MySQL
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password=""
    )
    cursor = conn.cursor()

    # Câu lệnh xóa database
    db_name = "math_grade3"
    cursor.execute(f"DROP DATABASE IF EXISTS {db_name}")
    
    print(f"Đã xóa database '{db_name}' thành công.")

    cursor.close()
    conn.close()
except mysql.connector.Error as err:
    print(f"Lỗi: {err}")

Đã xóa database 'math_grade3' thành công.


In [1]:
import mysql.connector

# Cấu hình kết nối (Dùng 127.0.0.1 để tránh lỗi treo)
config = { 'user': 'root', 'password': '', 'host': '127.0.0.1' }
DB_NAME = 'elearning_math_db'

try:
    conn = mysql.connector.connect(**config)
    cursor = conn.cursor()
    
    # 1. Reset Database (Xóa cũ tạo mới)
    cursor.execute(f"DROP DATABASE IF EXISTS {DB_NAME}")
    cursor.execute(f"CREATE DATABASE {DB_NAME} CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
    cursor.execute(f"USE {DB_NAME}")
    
    # 2. Bảng Giáo viên
    cursor.execute("""
    CREATE TABLE teachers (
        id INT AUTO_INCREMENT PRIMARY KEY,
        username VARCHAR(50) UNIQUE,
        password VARCHAR(50),
        full_name VARCHAR(100)
    )""")

    # 3. Bảng Học sinh
    cursor.execute("""
    CREATE TABLE students (
        id INT AUTO_INCREMENT PRIMARY KEY,
        full_name VARCHAR(100),
        parent_name VARCHAR(100),
        parent_phone VARCHAR(20) UNIQUE,
        teacher_id INT,
        FOREIGN KEY (teacher_id) REFERENCES teachers(id)
    )""")

    # 4. Bảng Bài học (Lessons)
    cursor.execute("""
    CREATE TABLE lessons (
        id INT AUTO_INCREMENT PRIMARY KEY,
        teacher_id INT,
        title VARCHAR(255),
        description TEXT, /* Vẫn giữ cột này trong DB dù UI không nhập, để dự phòng */
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (teacher_id) REFERENCES teachers(id)
    )""")

    # 5. Bảng Bộ đề (Exam Sets)
    cursor.execute("""
    CREATE TABLE exam_sets (
        id INT AUTO_INCREMENT PRIMARY KEY,
        lesson_id INT,
        title VARCHAR(255),
        status ENUM('draft', 'published') DEFAULT 'published',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (lesson_id) REFERENCES lessons(id) ON DELETE CASCADE
    )""")

    # 6. Bảng Câu hỏi
    cursor.execute("""
    CREATE TABLE questions (
        id INT AUTO_INCREMENT PRIMARY KEY,
        exam_set_id INT,
        question_text TEXT,
        svg_code LONGTEXT,
        options JSON,
        correct_answer VARCHAR(10),
        explanation TEXT,
        FOREIGN KEY (exam_set_id) REFERENCES exam_sets(id) ON DELETE CASCADE
    )""")

    # 7. Bảng Kết quả
    cursor.execute("""
    CREATE TABLE results (
        id INT AUTO_INCREMENT PRIMARY KEY,
        student_id INT,
        exam_set_id INT,
        score FLOAT,
        details JSON,
        submitted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (student_id) REFERENCES students(id),
        FOREIGN KEY (exam_set_id) REFERENCES exam_sets(id) ON DELETE CASCADE
    )""")

    # --- DỮ LIỆU MẪU ---
    cursor.execute("INSERT INTO teachers (username, password, full_name) VALUES ('gv1', '123', 'Cô Giáo Lan')")
    tid = cursor.lastrowid
    cursor.execute(f"INSERT INTO students (full_name, parent_name, parent_phone, teacher_id) VALUES ('Bé An', 'Anh Hùng', '0901234567', {tid})")

    conn.commit()
    print(">>> ĐÃ TẠO DATABASE THÀNH CÔNG! SẴN SÀNG CHẠY APP.")
    cursor.close()
    conn.close()

except mysql.connector.Error as err:
    print(f"Lỗi SQL: {err}")

>>> ĐÃ TẠO DATABASE THÀNH CÔNG! SẴN SÀNG CHẠY APP.


In [1]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    print("❌ Lỗi: Không tìm thấy API Key trong file .env")
else:
    genai.configure(api_key=API_KEY)
    print("--- DANH SÁCH MODEL CÓ THỂ DÙNG ---")
    try:
        for m in genai.list_models():
            if 'generateContent' in m.supported_generation_methods:
                print(f"✅ Tên model: {m.name}")
    except Exception as e:
        print(f"❌ Lỗi kết nối: {e}")

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


--- DANH SÁCH MODEL CÓ THỂ DÙNG ---
✅ Tên model: models/gemini-2.5-flash
✅ Tên model: models/gemini-2.5-pro
✅ Tên model: models/gemini-2.0-flash-exp
✅ Tên model: models/gemini-2.0-flash
✅ Tên model: models/gemini-2.0-flash-001
✅ Tên model: models/gemini-2.0-flash-exp-image-generation
✅ Tên model: models/gemini-2.0-flash-lite-001
✅ Tên model: models/gemini-2.0-flash-lite
✅ Tên model: models/gemini-2.0-flash-lite-preview-02-05
✅ Tên model: models/gemini-2.0-flash-lite-preview
✅ Tên model: models/gemini-exp-1206
✅ Tên model: models/gemini-2.5-flash-preview-tts
✅ Tên model: models/gemini-2.5-pro-preview-tts
✅ Tên model: models/gemma-3-1b-it
✅ Tên model: models/gemma-3-4b-it
✅ Tên model: models/gemma-3-12b-it
✅ Tên model: models/gemma-3-27b-it
✅ Tên model: models/gemma-3n-e4b-it
✅ Tên model: models/gemma-3n-e2b-it
✅ Tên model: models/gemini-flash-latest
✅ Tên model: models/gemini-flash-lite-latest
✅ Tên model: models/gemini-pro-latest
✅ Tên model: models/gemini-2.5-flash-lite
✅ Tên model: m

In [1]:
import mysql.connector
import random
import os
from dotenv import load_dotenv

# --- 1. CẤU HÌNH KẾT NỐI ---
load_dotenv() # Load thông tin từ file .env (để lấy DB_NAME, USER...)

db_config = {
    'host': os.getenv("DB_HOST", "127.0.0.1"),
    'user': os.getenv("DB_USER", "root"),
    'password': os.getenv("DB_PASSWORD", ""),
    'database': os.getenv("DB_NAME", "elearning_math_db")
}

# --- 2. DỮ LIỆU MẪU TIẾNG VIỆT ---
HO = ["Nguyễn", "Trần", "Lê", "Phạm", "Hoàng", "Huỳnh", "Phan", "Vũ", "Võ", "Đặng", "Bùi", "Đỗ"]
TEN_DEM_NAM = ["Văn", "Đức", "Minh", "Hữu", "Thanh", "Quốc", "Gia"]
TEN_DEM_NU = ["Thị", "Ngọc", "Thu", "Mai", "Phương", "Thanh", "Khánh"]
TEN = ["An", "Bình", "Châu", "Dũng", "Em", "Giang", "Hùng", "Hương", "Khánh", "Lan", "Minh", "Nam", "Nga", "Oanh", "Phúc", "Quân", "Sơn", "Tâm", "Uyên", "Vinh"]

def tao_ten_ngau_nhien():
    # Random giới tính (0: Nữ, 1: Nam) để chọn tên đệm
    gioi_tinh = random.choice([0, 1])
    ho = random.choice(HO)
    ten_dem = random.choice(TEN_DEM_NAM) if gioi_tinh == 1 else random.choice(TEN_DEM_NU)
    ten = random.choice(TEN)
    return f"{ho} {ten_dem} {ten}"

def tao_sdt_ngau_nhien():
    # Tạo SĐT dạng 09xxxxxxxx
    duoi = random.randint(10000000, 99999999)
    return f"09{duoi}"

def main():
    try:
        conn = mysql.connector.connect(**db_config)
        cur = conn.cursor()
        print("✅ Đã kết nối CSDL thành công!")

        # --- BƯỚC 1: TẠO 3 GIÁO VIÊN ---
        print("\n--- Đang tạo 3 Giáo viên ---")
        ds_giao_vien = [
            ("gv1", "123", "Cô Nguyễn Thu Hà"),
            ("gv2", "123", "Thầy Trần Văn Minh"),
            ("gv3", "123", "Cô Lê Thị Ngọc")
        ]
        
        teacher_ids = []

        for u, p, name in ds_giao_vien:
            # Kiểm tra xem đã tồn tại chưa để tránh lỗi trùng lặp
            cur.execute("SELECT id FROM teachers WHERE username = %s", (u,))
            res = cur.fetchone()
            
            if res:
                print(f"   -> Giáo viên {u} đã tồn tại (ID: {res[0]}). Bỏ qua.")
                teacher_ids.append(res[0])
            else:
                cur.execute("INSERT INTO teachers (username, password, full_name) VALUES (%s, %s, %s)", (u, p, name))
                new_id = cur.lastrowid
                teacher_ids.append(new_id)
                print(f"   -> Đã thêm: {name} (User: {u} / Pass: {p})")
        
        conn.commit()

        # --- BƯỚC 2: TẠO 20 HỌC SINH ---
        if not teacher_ids:
            print("❌ Lỗi: Không tìm thấy giáo viên nào để gán học sinh.")
            return

        print("\n--- Đang tạo 20 Học sinh ---")
        for i in range(20):
            full_name = tao_ten_ngau_nhien()
            parent_name = f"PH em {full_name.split()[-1]}" # Ví dụ: PH em Nam
            parent_phone = tao_sdt_ngau_nhien()
            
            # Chọn ngẫu nhiên 1 giáo viên để gán
            tid = random.choice(teacher_ids)
            
            # Thêm vào DB (Dùng IGNORE để nếu lỡ trùng SĐT thì bỏ qua)
            try:
                cur.execute("""
                    INSERT INTO students (full_name, parent_name, parent_phone, teacher_id) 
                    VALUES (%s, %s, %s, %s)
                """, (full_name, parent_name, parent_phone, tid))
                print(f"   [{i+1}/20] HS: {full_name} - SĐT: {parent_phone} -> Lớp GV ID: {tid}")
            except mysql.connector.Error as err:
                print(f"   [Lỗi thêm HS] {err}")

        conn.commit()
        print("\n🎉 HOÀN TẤT! Đã giả lập dữ liệu thành công.")
        
    except mysql.connector.Error as err:
        print(f"❌ Lỗi kết nối CSDL: {err}")
    finally:
        if conn.is_connected():
            cur.close()
            conn.close()

if __name__ == "__main__":
    main()

✅ Đã kết nối CSDL thành công!

--- Đang tạo 3 Giáo viên ---
   -> Giáo viên gv1 đã tồn tại (ID: 1). Bỏ qua.
   -> Đã thêm: Thầy Trần Văn Minh (User: gv2 / Pass: 123)
   -> Đã thêm: Cô Lê Thị Ngọc (User: gv3 / Pass: 123)

--- Đang tạo 20 Học sinh ---
   [1/20] HS: Võ Thanh Quân - SĐT: 0964779874 -> Lớp GV ID: 2
   [2/20] HS: Đặng Minh Uyên - SĐT: 0987287218 -> Lớp GV ID: 1
   [3/20] HS: Trần Mai Bình - SĐT: 0962761857 -> Lớp GV ID: 3
   [4/20] HS: Phạm Thị Lan - SĐT: 0976924451 -> Lớp GV ID: 2
   [5/20] HS: Nguyễn Thị Lan - SĐT: 0920719497 -> Lớp GV ID: 1
   [6/20] HS: Võ Thanh Phúc - SĐT: 0930759300 -> Lớp GV ID: 3
   [7/20] HS: Vũ Thanh Dũng - SĐT: 0995767827 -> Lớp GV ID: 1
   [8/20] HS: Bùi Khánh Oanh - SĐT: 0928271867 -> Lớp GV ID: 3
   [9/20] HS: Huỳnh Thanh Lan - SĐT: 0997546028 -> Lớp GV ID: 3
   [10/20] HS: Lê Văn Hùng - SĐT: 0996811227 -> Lớp GV ID: 1
   [11/20] HS: Đỗ Gia Uyên - SĐT: 0910038859 -> Lớp GV ID: 2
   [12/20] HS: Đặng Thị Dũng - SĐT: 0920192520 -> Lớp GV ID: 3
   

In [2]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv()

db_config = {
    'host': os.getenv("DB_HOST", "127.0.0.1"),
    'user': os.getenv("DB_USER", "root"),
    'password': os.getenv("DB_PASSWORD", ""),
    'database': os.getenv("DB_NAME", "elearning_math_db")
}

def update_database():
    try:
        conn = mysql.connector.connect(**db_config)
        cur = conn.cursor()
        
        print("🛠 Đang cập nhật cấu trúc bảng 'students'...")
        
        # Thêm cột password nếu chưa có
        try:
            cur.execute("ALTER TABLE students ADD COLUMN password VARCHAR(255) NULL AFTER parent_phone")
            print("✅ Đã thêm cột 'password' thành công!")
        except mysql.connector.Error as err:
            if err.errno == 1060: # Mã lỗi Duplicate column name
                print("ℹ️ Cột 'password' đã tồn tại. Không cần thêm.")
            else:
                print(f"❌ Lỗi SQL: {err}")

        conn.commit()
        print("🎉 Cập nhật CSDL hoàn tất!")
        
    except mysql.connector.Error as err:
        print(f"❌ Lỗi kết nối CSDL: {err}")
    finally:
        if 'conn' in locals() and conn.is_connected():
            conn.close()

if __name__ == "__main__":
    update_database()

🛠 Đang cập nhật cấu trúc bảng 'students'...
✅ Đã thêm cột 'password' thành công!
🎉 Cập nhật CSDL hoàn tất!


In [3]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv()

db_config = {
    'host': os.getenv("DB_HOST", "127.0.0.1"),
    'user': os.getenv("DB_USER", "root"),
    'password': os.getenv("DB_PASSWORD", ""),
    'database': os.getenv("DB_NAME", "elearning_math_db")
}

def update_db():
    conn = mysql.connector.connect(**db_config)
    cur = conn.cursor()
    try:
        # Thêm cột created_at cho bảng results để thống kê theo thời gian
        print("🛠 Đang kiểm tra bảng results...")
        cur.execute("ALTER TABLE results ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP")
        print("✅ Đã thêm cột 'created_at' thành công!")
    except mysql.connector.Error as err:
        print(f"ℹ️ Thông báo: {err}") # Thường là lỗi cột đã tồn tại (không sao cả)
    finally:
        conn.close()

if __name__ == "__main__":
    update_db()

🛠 Đang kiểm tra bảng results...
✅ Đã thêm cột 'created_at' thành công!


In [ ]:
# Cell 1: Import thư viện và Cấu hình
import mysql.connector
import os
from dotenv import load_dotenv

# 1. Xác định vị trí file .env (Nằm ở thư mục cha của thư mục database)
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
env_path = os.path.join(parent_dir, '.env')

# 2. Load biến môi trường
if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"✅ Đã tìm thấy và load file .env tại: {env_path}")
else:
    # Fallback: Thử load từ đường dẫn hiện tại nếu chạy từ root
    load_dotenv() 
    print("⚠️ Không thấy .env ở thư mục cha, đang thử load từ thư mục hiện tại...")

# 3. Lấy thông tin cấu hình
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_NAME = os.getenv("DB_NAME", "elearning_math_db")

print(f"📡 Cấu hình: Host={DB_HOST}, User={DB_USER}, DB={DB_NAME}")

# Cell 2: Hàm kết nối và Hàm thực thi SQL an toàn
def get_connection():
    try:
        conn = mysql.connector.connect(
            host=DB_HOST,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        return conn
    except mysql.connector.Error as err:
        print(f"❌ Lỗi kết nối: {err}")
        return None

def execute_query(conn, query, params=None):
    cursor = conn.cursor()
    try:
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        print("   ✅ Thành công.")
    except mysql.connector.Error as err:
        # Mã 1060: Duplicate column name (Cột đã tồn tại)
        # Mã 1050: Table already exists (Bảng đã tồn tại)
        if err.errno == 1060: 
            print("   ⚠️ Cột này đã tồn tại, bỏ qua.")
        elif err.errno == 1050:
            print("   ⚠️ Bảng này đã tồn tại, bỏ qua.")
        elif err.errno == 1061:
             print("   ⚠️ Key/Index đã tồn tại, bỏ qua.")
        else:
            print(f"   ❌ Lỗi SQL: {err}")
    finally:
        cursor.close()

# Cell 3: BẮT ĐẦU CẬP NHẬT DATABASE
conn = get_connection()

if conn:
    print("\n🚀 BẮT ĐẦU CẬP NHẬT DATABASE CHO TÍNH NĂNG MỚI...")
    
    # 1. Tạo bảng Classes (Lớp học)
    print("\n1️⃣ Tạo bảng 'classes'...")
    sql_create_classes = """
    CREATE TABLE IF NOT EXISTS classes (
        id INT AUTO_INCREMENT PRIMARY KEY,
        teacher_id INT NOT NULL,
        class_name VARCHAR(100) NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (teacher_id) REFERENCES teachers(id) ON DELETE CASCADE
    );
    """
    execute_query(conn, sql_create_classes)

    # 2. Thêm cột class_id vào bảng students
    print("\n2️⃣ Thêm cột 'class_id' vào bảng 'students'...")
    sql_add_class_id = "ALTER TABLE students ADD COLUMN class_id INT NULL;"
    execute_query(conn, sql_add_class_id)

    # 3. Thêm cột dob (Ngày sinh) vào bảng students
    print("\n3️⃣ Thêm cột 'dob' (Ngày sinh) vào bảng 'students'...")
    sql_add_dob = "ALTER TABLE students ADD COLUMN dob DATE NULL;"
    execute_query(conn, sql_add_dob)

    # 4. Tạo Khóa ngoại (Foreign Key) cho class_id
    # Lưu ý: Cần try/except riêng vì nếu chạy lần 2 sẽ báo lỗi duplicate key name
    print("\n4️⃣ Tạo khóa ngoại liên kết Student -> Class...")
    try:
        cursor = conn.cursor()
        cursor.execute("ALTER TABLE students ADD CONSTRAINT fk_student_class FOREIGN KEY (class_id) REFERENCES classes(id) ON DELETE SET NULL;")
        conn.commit()
        print("   ✅ Đã tạo khóa ngoại thành công.")
    except mysql.connector.Error as err:
        if err.errno == 1061 or 'Duplicate' in str(err):
            print("   ⚠️ Khóa ngoại đã tồn tại, bỏ qua.")
        else:
            print(f"   ❌ Lỗi tạo khóa ngoại: {err}")

    # 5. DATA MIGRATION (Chuyển đổi dữ liệu cũ)
    # Tạo một lớp mặc định để chứa các học sinh cũ chưa có lớp
    print("\n5️⃣ Xử lý dữ liệu cũ (Data Migration)...")
    
    # Lấy ID giáo viên đầu tiên để tạo lớp mẫu
    cursor = conn.cursor(dictionary=True)
    cursor.execute("SELECT id FROM teachers LIMIT 1")
    teacher = cursor.fetchone()
    
    if teacher:
        t_id = teacher['id']
        # Kiểm tra xem đã có lớp nào chưa
        cursor.execute("SELECT id FROM classes WHERE teacher_id=%s LIMIT 1", (t_id,))
        existing_class = cursor.fetchone()
        
        class_id_to_assign = None
        
        if not existing_class:
            print(f"   ℹ️ Chưa có lớp nào, tạo lớp mẫu 'Lớp 2025' cho GV ID {t_id}...")
            cursor.execute("INSERT INTO classes (teacher_id, class_name) VALUES (%s, 'Lớp 2025 (Mặc định)')", (t_id,))
            conn.commit()
            class_id_to_assign = cursor.lastrowid
        else:
            class_id_to_assign = existing_class['id']
            
        # Cập nhật tất cả học sinh cũ chưa có lớp vào lớp này
        if class_id_to_assign:
            print(f"   🔄 Đang gán học sinh cũ vào Class ID {class_id_to_assign}...")
            cursor.execute("UPDATE students SET class_id=%s WHERE class_id IS NULL", (class_id_to_assign,))
            conn.commit()
            print("   ✅ Hoàn tất gán lớp.")
    else:
        print("   ⚠️ Không tìm thấy giáo viên nào để tạo lớp mẫu.")

    conn.close()
    print("\n🎉🎉🎉 CẬP NHẬT DATABASE HOÀN TẤT! SẴN SÀNG CHẠY APP.")
else:
    print("❌ Không thể cập nhật do lỗi kết nối.")

NameError: name '__file__' is not defined